### Analysis and Evaluation

Human tutors and students in
    `results/annotator/v13/structure_labels_gold_anthropic_test_scaffolding.json`

Join each annotated moment to `situation_label_agg` from `data/ground_truth_hybrid/` on `(conversation_id, turn_start, turn_end)`, and then ask
whether the tutor's `action_label` matches what the situation called for. 

In [1]:
import json
import glob
import os
from collections import Counter

REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
STRUCTURE_LABELS = [os.path.join(
    REPO, "results/annotator/v13/structure_labels_gold_anthropic_test_scaffolding.json"
    ), 
    os.path.join(
        REPO, "results/annotator/v13/structure_labels_gold_anthropic_scaffolding.json"
    )
]
GROUND_TRUTH_DIR = os.path.join(REPO, "data/ground_truth_hybrid")

In [2]:
# --- Load the human situation labels from ground_truth_hybrid ---
gt_situation = {}
for f in glob.glob(os.path.join(GROUND_TRUTH_DIR, "*.json")):
    cid = os.path.basename(f)[:-5]
    d = json.load(open(f))
    for km in d["key_moments"]:
        gt_situation[(cid, km["turn_start"], km["turn_end"])] = km.get("situation_label_agg")

# --- load v13 structure labels (claude-opus-4-8 annotator output) ---
rows = []
for structure_labels_file in STRUCTURE_LABELS:
    sl = json.load(open(structure_labels_file))
    print(f"Source: {sl['source']} | model: {sl['model']} | version: {sl['version']}")
    print(f"{sl['total_conversations']} conversations, {sl['total_annotations']} annotations\n")
    # --- Join: attach the human situation label to each v13 annotation ---

    for cid_full, entry in sl["results"].items():
        cid = cid_full.split("_")[-1]  # last UUID segment == ground-truth filename
        for ann in entry["annotations"]:
            key = (cid, ann["turn_start"], ann["turn_end"])
            rows.append(
                {
                    "situation": gt_situation[key],  # human label
                    "action": ann["action_label"],       # tutor action direction
                    "result": ann["result_label"],       # student outcome
                }
            )

n_matched = sum(1 for r in rows if r["situation"] is not None)
print(f"Joined {len(rows)} moments ({n_matched} matched a human situation label)\n")
print("situation_label_agg :", dict(Counter(r["situation"] for r in rows)))
print("action_label        :", dict(Counter(r["action"] for r in rows)))
print("result_label        :", dict(Counter(r["result"] for r in rows)))


Source: gold_truth | model: claude-opus-4-8 | version: v13
70 conversations, 670 annotations

Source: gold_truth | model: claude-opus-4-8 | version: v13
52 conversations, 866 annotations

Joined 1536 moments (1536 matched a human situation label)

situation_label_agg : {'mixed': 385, 'scaffolding': 719, 'neither': 130, 'rigor': 293, 'both': 2, 'unknown': 7}
action_label        : {'scaffolding': 762, 'both': 431, 'neither': 149, 'rigor': 194}
result_label        : {'pos': 1007, 'neg': 503, 'no_evidence': 26}


In [3]:
def pct(subset, field, values):
    """Count rows whose `field` is in `values`, return (count, total, percent)."""
    c = sum(1 for r in subset if r[field] in values)
    total = len(subset)
    return c, total, (100 * c / total if total else 0.0)


scaffolding_sit = [r for r in rows if r["situation"] == "scaffolding"]
rigor_sit = [r for r in rows if r["situation"] == "rigor"]

# Do tutors take the action the situation calls for?
# A moment "has a scaffolding action" if action_label is scaffolding OR both
# (the tutor scaffolded AND pushed rigor); same logic for rigor.
print("Scaffolding-appropriate situations (situation_label_agg == 'scaffolding')")
print("  %d/%d = %.1f%% have a scaffolding action" % pct(scaffolding_sit, "action", {"scaffolding", "both"}))
print()
print("Rigor-appropriate situations (situation_label_agg == 'rigor')")
print("  %d/%d = %.1f%% have a rigor action" % pct(rigor_sit, "action", {"rigor", "both"}))
print()

# Student outcome across all moments.
print("Student outcome across all %d moments" % len(rows))
print("  %d/%d = %.1f%% positive" % pct(rows, "result", {"pos"}))
print("  %d/%d = %.1f%% negative" % pct(rows, "result", {"neg"}))
print("  %d/%d = %.1f%% no evidence" % pct(rows, "result", {"no_evidence"}))


Scaffolding-appropriate situations (situation_label_agg == 'scaffolding')
  689/719 = 95.8% have a scaffolding action

Rigor-appropriate situations (situation_label_agg == 'rigor')
  188/293 = 64.2% have a rigor action

Student outcome across all 1536 moments
  1007/1536 = 65.6% positive
  503/1536 = 32.7% negative
  26/1536 = 1.7% no evidence
